*0.3 Classical NLP*

# Tokenization: tiktoken

**The situation.** Finance wants cost per ticket before the call is made, and the context limit must never be exceeded. The team estimates tokens as "words × 1.3". A batch of German tickets blows past the limit and fails; a batch of code snippets costs twice the estimate.

**tiktoken.** OpenAI's tokenizer library. It contains the exact BPE vocabularies the models use, so the count it gives is the count you are billed for. Counting is local and takes microseconds — do it before every call.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

In [2]:
import tiktoken

encoding = tiktoken.encoding_for_model(MODEL)
samples = {
    "English": "The customer was charged twice for the order and wants a refund.",
    "German": "Dem Kunden wurde die Bestellung doppelt berechnet und er möchte eine Rückerstattung.",
    "code": "for ticket in tickets:\n    if ticket.status == 'open':\n        escalate(ticket)",
}
print(f"{'sample':<8}{'words':>7}{'tokens':>8}{'tokens/word':>13}")
ratios = {}
for label, text in samples.items():
    words = len(text.split())
    tokens = len(encoding.encode(text))
    ratios[label] = tokens / words
    print(f"{label:<8}{words:>7}{tokens:>8}{ratios[label]:>13.2f}")
print(
    "pieces of 'Rückerstattung':", encoding.decode_tokens_bytes(encoding.encode("Rückerstattung"))
)
assert ratios["German"] > ratios["English"]

sample    words  tokens  tokens/word
English      12      13         1.08
German       12      17         1.42
code          9      17         1.89
pieces of 'Rückerstattung': [b'R', b'\xc3\xbcck', b'erst', b'attung']


**Reading the output.** English is close to the "1.3 per word" folk estimate. German and code are well above it — one German word becomes several pieces. That is the difference between a batch that fits and one that fails.

**Cost and limit check before the call.** This is the function that belongs in front of every request.

In [3]:
PRICE_PER_MILLION_INPUT = 0.15  # USD, gpt-4o-mini input
CONTEXT_LIMIT = 128_000


def check_request(messages: list[dict], max_output_tokens: int) -> dict:
    tokens = 0
    for message in messages:
        tokens += 4 + len(encoding.encode(message["content"]))  # ~4 tokens of per-message overhead
    return {
        "input_tokens": tokens,
        "fits": tokens + max_output_tokens <= CONTEXT_LIMIT,
        "input_cost_usd": round(tokens * PRICE_PER_MILLION_INPUT / 1e6, 6),
    }


long_ticket = samples["German"] * 300
print(check_request([{"role": "user", "content": samples["English"]}], max_output_tokens=500))
print(check_request([{"role": "user", "content": long_ticket}], max_output_tokens=500))
assert check_request([{"role": "user", "content": long_ticket}], 500)["fits"]

{'input_tokens': 17, 'fits': True, 'input_cost_usd': 3e-06}
{'input_tokens': 5104, 'fits': True, 'input_cost_usd': 0.000766}


**The rule to remember.** Count tokens with the model's own tokenizer, locally, before every call. Words are not tokens; the ratio depends on language and content.

| Use it when | Don't when | Instead use |
|---|---|---|
| any OpenAI model: cost estimates, truncation, context checks | other providers — their tokenizers differ | the provider's tokenizer (`AutoTokenizer` for open models; Anthropic's count-tokens endpoint) |

**Watch out**
- Per-message overhead (role, separators) is a few tokens; the exact number changes by model version. Leave headroom.
- Images and tool schemas count too, and not through this function.
- `encoding_for_model` needs a model name tiktoken knows; for brand-new models fall back to `get_encoding("o200k_base")`.